# Linear Systems

## What's covered

- Revisiting `Ax = b` with the vocabulary built up in the last three notebooks
- **Gaussian elimination** — the algorithm behind every solver, in one paragraph
- **Rank** — the single number that decides solvability and uniqueness
- **Column space** — when does `Ax = b` have *any* solution?
- **Null space** — when does `Ax = b` have *one* solution vs many?
- The **four fundamental subspaces** of a matrix (Strang's picture)
- **Overdetermined** vs **underdetermined** systems, and a first look at **least squares**
- Where this appears in ML — normal equations, ridge regression, pseudoinverse


## Revisiting Ax = b with new vocabulary

In notebook 1 we met `Ax = b` with two interpretations: a system of equations, and the geometry of intersecting lines / planes. With the column picture from notebook 4 we now have a sharper third interpretation:

> **`Ax = b` asks: can we write `b` as a linear combination of the columns of `A`?**

If yes, the weights of that combination *are* `x`. If no, there is no solution.

That single sentence connects everything in this notebook. The three classical cases of the system — unique, none, infinite — fall out of two questions:

1. **Solvability.** Is `b` in the **column space** of `A`?
2. **Uniqueness.** Is the **null space** of `A` just the zero vector?

| `b` in col space? | null space trivial? | Outcome |
|---|---|---|
| yes | yes | exactly one solution |
| yes | no  | infinitely many solutions |
| no  | —   | no solution (least squares finds the best approximate one) |

The rest of this notebook is just unpacking that table.


## Gaussian elimination, in one paragraph

Every numerical solver, under the hood, does the same trick you learned in school: **scale and subtract rows until the system is upper triangular, then back-substitute.** The three legal moves — swap rows, scale a row by a non-zero constant, add a multiple of one row to another — never change the solution set, only its presentation. The endpoint is **row echelon form** (or **reduced row echelon form**, RREF, if you also scale leading entries to 1 and zero them out above as well as below).

You almost never run elimination by hand in ML. But two ideas survive every encounter with it:

- The number of non-zero rows in the echelon form is the **rank**.
- The columns containing the leading 1's (the **pivot columns**) form a basis for the column space.

In code, you call `np.linalg.solve(A, b)` for square invertible `A`, and `np.linalg.lstsq(A, b)` when you want the least-squares answer for any shape. We will use both.


In [ ]:
import numpy as np

# A square, invertible system
A = np.array([[2, 1, 1],
              [1, 3, 2],
              [1, 0, 1]], dtype=float)
b = np.array([8, 13, 3], dtype=float)

x = np.linalg.solve(A, b)
print("x      =", x)
print("A @ x  =", A @ x, "  vs  b =", b)


## Rank, column space, null space — the trio

Three quantities tell you everything about `Ax = b`. They are all properties of `A` alone — `b` does not enter yet.

**Rank.** The **rank** of `A` is the dimension of the column space — equivalently, the maximum number of linearly independent columns (or rows; column rank always equals row rank). For an `m × n` matrix, rank can be at most `min(m, n)`. A matrix that achieves this maximum is called **full rank**.

**Column space (`Col(A)`).** The span of the columns of `A`. It is a subspace of `R^m`. `Ax = b` has a solution if and only if `b ∈ Col(A)`. Its dimension is `rank(A)`.

**Null space (`Null(A)`).** The set of all `x ∈ R^n` with `Ax = 0`. It is a subspace of `R^n`. Its dimension is `n - rank(A)` — a fact known as the **rank-nullity theorem**:

$$
\dim(\text{Col}(A)) + \dim(\text{Null}(A)) = n
$$

(`n` is the number of columns of `A`, which is the dimension of the input space.)

**The connection to solutions of `Ax = b`.** If `x_p` is *any* particular solution and `x_0` is in the null space, then `x_p + x_0` is also a solution (because `A x_0 = 0`). So:

- Solution exists ⇔ `b ∈ Col(A)`.
- Solution is unique ⇔ `Null(A) = {0}` ⇔ `rank(A) = n`.

Memorize these. They show up in every ML interview that touches "why is `X^T X` invertible?"


In [ ]:
# Three matrices, three rank profiles
A_full = np.array([[2, 1], [1, 3]])                     # 2x2, full rank
A_tall = np.array([[1, 0], [0, 1], [1, 1]])             # 3x2, rank 2 (full column rank)
A_rdef = np.array([[1, 2, 3], [2, 4, 6]])               # 2x3, rank 1 — columns all proportional

for name, M in [("A_full", A_full), ("A_tall", A_tall), ("A_rdef", A_rdef)]:
    r = np.linalg.matrix_rank(M)
    print(f"{name}: shape={M.shape}, rank={r}, dim(null space)={M.shape[1] - r}")

# Test whether a particular b lies in the column space of A_tall
b_in  = A_tall @ np.array([3, 4])   # by construction, this b is in Col(A_tall)
b_out = np.array([1, 2, 9])         # arbitrary — probably not in Col(A_tall)

def in_col_space(A, b, tol=1e-8):
    x, *_ = np.linalg.lstsq(A, b, rcond=None)
    return np.allclose(A @ x, b, atol=tol)

print("\nb_in  in Col(A_tall)?", in_col_space(A_tall, b_in))
print("b_out in Col(A_tall)?", in_col_space(A_tall, b_out))


In [ ]:
# Find a basis for the null space using SVD
# (the columns of V corresponding to zero singular values span Null(A))
def null_space(A, tol=1e-10):
    U, s, Vt = np.linalg.svd(A)
    n = A.shape[1]
    zero_singular = np.concatenate([s, np.zeros(max(0, n - len(s)))]) < tol
    return Vt[zero_singular].T

# A_rdef has rank 1, so its null space is 2-dimensional
N = null_space(A_rdef)
print("Null space basis of A_rdef (columns):\n", N)
print("Sanity check — A @ null vector should be 0:\n", A_rdef @ N)


## The four fundamental subspaces

Every `m × n` matrix `A` defines four subspaces that together describe everything the matrix does. Gilbert Strang's diagram organizes them into two pairs:

**In the input space `R^n`:**

- **Row space `Row(A) = Col(A^T)`** — span of the rows of `A`. Dimension: `rank(A)`.
- **Null space `Null(A)`** — what gets sent to zero. Dimension: `n - rank(A)`.

**In the output space `R^m`:**

- **Column space `Col(A)`** — what `A` can reach. Dimension: `rank(A)`.
- **Left null space `Null(A^T)`** — vectors orthogonal to `Col(A)`. Dimension: `m - rank(A)`.

Two beautiful facts make the picture click:

1. **Row space and null space are orthogonal complements in `R^n`.** Every vector in `R^n` splits cleanly into a piece in the row space and a piece in the null space, and the two pieces are perpendicular.
2. **Column space and left null space are orthogonal complements in `R^m`.** Same story on the output side.

A small ASCII version of the picture:

```
        R^n                       R^m
   +--------------+           +--------------+
   |  Row(A)      |  -- A --> |  Col(A)      |
   |   (dim r)    |           |   (dim r)    |
   |--------------|           |--------------|
   |  Null(A)     |  -- A --> |  {0}         |
   |  (dim n-r)   |           |              |
   +--------------+           +--------------+
                              |  Null(A^T)   |
                              |  (dim m-r)   |
                              +--------------+
```

The map `A` shrinks the row space into the column space (bijectively!), and crushes the null space to zero. Everything in the left null space is unreachable.

Why this matters in ML: **least squares projects `b` onto the column space**, because that is the closest point that `Ax` can actually reach. The residual `b - Ax` lives in the left null space — orthogonal to the column space. We will use this directly in the orthogonality notebook.


## Overdetermined and underdetermined systems

In ML, square invertible systems are the exception. You almost always have one of:

**Overdetermined** — `m > n`, more equations than unknowns. Linear regression with thousands of data points and a handful of features. Typically `b ∉ Col(A)` and there is **no exact solution**. We instead minimize `||Ax - b||_2^2`. This is **least squares**, and the optimal `x` solves the **normal equations**:

$$
A^T A \, \mathbf{x} = A^T \mathbf{b}
$$

Geometrically: project `b` onto `Col(A)`, then solve `Ax = (\text{projection of } b)`. The next notebook (orthogonality) makes this picture explicit.

**Underdetermined** — `m < n`, fewer equations than unknowns. Modern ML loves this regime: more features than samples, or more parameters than training points. Now `Null(A) ≠ {0}`, so any solution can have anything from the null space added to it without breaking the equation. There is an **infinite family of solutions**, and you have to pick one. The standard choices:

- **Minimum-norm solution** — pick the `x` with the smallest `||x||_2`. This is what `np.linalg.lstsq` returns. Closed form: `x = A^T (A A^T)^{-1} b`.
- **Regularization** — add a penalty (ridge: `λ ||x||_2^2`; lasso: `λ ||x||_1`) to single out one solution from the family.

Both least squares and the minimum-norm solution are captured by the **pseudoinverse** `A^+`, computed via SVD. We will build it carefully in the SVD notebook.


In [ ]:
# Overdetermined: linear regression on a small synthetic dataset
rng = np.random.default_rng(0)
n_samples, n_features = 50, 2
X = rng.normal(size=(n_samples, n_features))
true_w = np.array([2.0, -1.0])
y = X @ true_w + 0.3 * rng.normal(size=n_samples)   # noisy targets — no exact solution

# Least squares via the normal equations (don't do this for real, but it's instructive)
w_normal = np.linalg.solve(X.T @ X, X.T @ y)

# Least squares via lstsq (the right way numerically)
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)

print("true weights      :", true_w)
print("normal equations w:", w_normal)
print("lstsq w           :", w_lstsq)

# Residual lives in the left null space of X (orthogonal to columns of X)
residual = y - X @ w_lstsq
print("X^T @ residual    :", X.T @ residual, "  <- approx zero — residual is orthogonal to Col(X)")


In [ ]:
# Underdetermined: 2 equations, 3 unknowns. Infinite solutions; lstsq returns the min-norm one.
A = np.array([[1, 1, 1],
              [1, 2, 3]], dtype=float)
b = np.array([6, 14], dtype=float)

x_min, *_ = np.linalg.lstsq(A, b, rcond=None)
print("min-norm solution:", x_min, "  ||x|| =", np.linalg.norm(x_min))

# Another valid solution = x_min + (any null-space vector)
N = null_space(A)
x_other = x_min + 2.5 * N[:, 0]
print("another solution :", x_other, "  ||x|| =", np.linalg.norm(x_other), "  (larger norm)")
print("A @ x_other      :", A @ x_other, "  vs  b =", b)


## Where this appears in ML

This notebook is the load-bearing one for most "applied" linear algebra in ML. Almost every model-fitting story is hidden in here:

- **Linear regression (ordinary least squares).** Solve `(X^T X) w = X^T y`. Closed form when `X^T X` is invertible; least squares more generally. Geometrically: project `y` onto `Col(X)`.
- **Ridge regression.** Solve `(X^T X + λI) w = X^T y`. The `λI` term forces full rank — it fixes the underdetermined / multicollinearity case (`Null(X) ≠ {0}`) by picking a unique solution.
- **Why `X^T X` may be singular.** Multicollinearity = columns of `X` are linearly dependent = `rank(X) < n` = non-trivial null space = non-unique solution. The same story, four ways.
- **Lasso regression.** Same as ridge in spirit but with an L1 penalty, which selects sparse solutions inside the null space.
- **Pseudoinverse `X^+`.** The universal solver: `x = X^+ b` gives the minimum-norm least-squares solution, no matter the shape or rank of `X`. Built from SVD in the last notebook.
- **Solving for backpropagation in linear layers.** Each gradient step solves a tiny linear system, conceptually.
- **Recommender systems (matrix completion).** Given partial observations of a low-rank matrix, recover the rest by solving an underdetermined system with a low-rank constraint.
- **PageRank, spectral clustering.** Eigenvectors are solutions to `(A - λI) v = 0` — a homogeneous linear system. The null space of `(A - λI)` is the eigenspace for eigenvalue `λ`.

Next notebook: **orthogonality** — we hand-waved at "project `y` onto `Col(X)`." Now we make that picture precise, derive the normal equations from it, and meet Gram-Schmidt and the QR decomposition.
